# Benchmark Completo — Transfer Learning no CIFAR-10
**Versão Professor · Dataset COMPLETO 50.000 imagens · 10 épocas · GPU**

Roda os 5 experimentos com o dataset completo do CIFAR-10. Resultados mais realistas e robustos do que o subset de 5k.

| # | Estratégia | Params treináveis | Tempo estimado (GPU T4) |
|---|---|---|---|
| 1 | Scratch Padrão | 11.2M | ~15–20 min |
| 2 | Scratch + Augmentation | 11.2M | ~15–20 min |
| 3 | Feature Extraction | 5.1K | ~8–12 min |
| 4 | Fine-Tuning Parcial (layer4) | 2.6M | ~15–20 min |
| 5 | Fine-Tuning Completo | 11.2M | ~20–25 min |

> ⏱️ **Tempo total estimado com GPU T4: ~80–100 min.** Recomenda-se rodar em segundo plano ou durante uma pausa.

## ▶️ Passo 0 — Ativar a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**

Execute a célula abaixo para confirmar. Sem GPU este notebook levaria mais de 10 horas.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative antes de continuar!')
    raise SystemExit('Interrompido: GPU obrigatória para este notebook.')

✅ GPU ativa: Tesla T4


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)

NUM_EPOCHS = 10
N_TRAIN    = 50000   # dataset completo
N_VAL      = 10000   # validação completa
BATCH      = 128     # batch maior aproveita melhor a GPU com dataset grande

results = {}
print(f'Config: {N_TRAIN} treino | {N_VAL} val | {NUM_EPOCHS} épocas | batch={BATCH} | device={device}')

Config: 50000 treino | 10000 val | 10 épocas | batch=128 | device=cuda


In [ ]:
transform_std = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset completo — sem subset
train_full_std = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform_std)
train_full_aug = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform_aug)
val_full       = datasets.CIFAR10(root='data', train=False, download=True, transform=transform_std)

train_loader_std = DataLoader(train_full_std, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
train_loader_aug = DataLoader(train_full_aug, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader       = DataLoader(val_full,       batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Dataset completo: {len(train_full_std)} treino | {len(val_full)} val')

100%|██████████| 170M/170M [00:05<00:00, 28.9MB/s]


✅ Dataset completo: 50000 treino | 10000 val


In [ ]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

def run_experiment(name, model, optimizer, train_loader, n_epochs=NUM_EPOCHS):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'  Parâmetros treináveis: {trainable:,}')
    print(f'{"="*55}')
    t0 = time.time()
    acc = 0.0
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer)
        acc  = evaluate(model, val_loader)
        elapsed_ep = time.time() - t0
        print(f'  Época {epoch:>2}/{n_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | {elapsed_ep/60:.1f}min acumulado')
    elapsed = time.time() - t0
    print(f'  ✅ {elapsed/60:.1f} min total | Acurácia final: {acc*100:.2f}%')
    results[name] = {'acc': acc * 100, 'time_min': elapsed / 60, 'params': trainable}
    return model

print('✅ Funções auxiliares definidas.')

✅ Funções auxiliares definidas.




```
# Isto está formatado como código
```

## Experimento 1 — Scratch Padrão
ResNet-18 com pesos aleatórios, sem nenhum conhecimento prévio.

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
run_experiment('Scratch Padrão', m, opt, train_loader_std)


  Scratch Padrão
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 1.5033 | Val Acc: 53.40% | 2.7min acumulado
  Época  2/10 | Loss: 0.9924 | Val Acc: 62.12% | 5.3min acumulado
  Época  3/10 | Loss: 0.7169 | Val Acc: 71.90% | 8.0min acumulado
  Época  4/10 | Loss: 0.5442 | Val Acc: 69.30% | 10.7min acumulado
  Época  5/10 | Loss: 0.3996 | Val Acc: 74.20% | 13.4min acumulado
  Época  6/10 | Loss: 0.2762 | Val Acc: 72.49% | 16.1min acumulado
  Época  7/10 | Loss: 0.1670 | Val Acc: 74.50% | 18.8min acumulado
  Época  8/10 | Loss: 0.1040 | Val Acc: 60.47% | 21.5min acumulado
  Época  9/10 | Loss: 0.0562 | Val Acc: 74.52% | 24.2min acumulado
  Época 10/10 | Loss: 0.0242 | Val Acc: 75.97% | 26.9min acumulado
  ✅ 26.9 min total | Acurácia final: 75.97%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 2 — Scratch + Data Augmentation
Mesmo modelo do zero com flip e rotação no treino.

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
run_experiment('Scratch + Augmentation', m, opt, train_loader_aug)


  Scratch + Augmentation
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 1.5666 | Val Acc: 49.19% | 2.7min acumulado
  Época  2/10 | Loss: 1.1221 | Val Acc: 63.09% | 5.4min acumulado
  Época  3/10 | Loss: 0.8921 | Val Acc: 72.53% | 8.1min acumulado
  Época  4/10 | Loss: 0.7490 | Val Acc: 74.21% | 10.8min acumulado
  Época  5/10 | Loss: 0.6513 | Val Acc: 74.73% | 13.5min acumulado
  Época  6/10 | Loss: 0.5889 | Val Acc: 79.66% | 16.2min acumulado
  Época  7/10 | Loss: 0.5263 | Val Acc: 80.63% | 18.9min acumulado
  Época  8/10 | Loss: 0.4810 | Val Acc: 80.94% | 21.6min acumulado
  Época  9/10 | Loss: 0.4417 | Val Acc: 80.87% | 24.4min acumulado
  Época 10/10 | Loss: 0.4083 | Val Acc: 83.42% | 27.1min acumulado
  ✅ 27.1 min total | Acurácia final: 83.42%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 3 — Feature Extraction
Backbone congelado, treina apenas `fc` (5.130 parâmetros).

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.fc.parameters(), lr=0.01, momentum=0.9)
run_experiment('Feature Extraction', m, opt, train_loader_std)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 181MB/s]



  Feature Extraction
  Parâmetros treináveis: 5,130
  Época  1/10 | Loss: 0.7402 | Val Acc: 79.10% | 1.8min acumulado
  Época  2/10 | Loss: 0.5941 | Val Acc: 79.35% | 3.6min acumulado
  Época  3/10 | Loss: 0.5773 | Val Acc: 79.53% | 5.4min acumulado
  Época  4/10 | Loss: 0.5581 | Val Acc: 79.17% | 7.2min acumulado
  Época  5/10 | Loss: 0.5527 | Val Acc: 80.58% | 9.0min acumulado
  Época  6/10 | Loss: 0.5503 | Val Acc: 80.48% | 10.8min acumulado
  Época  7/10 | Loss: 0.5454 | Val Acc: 80.17% | 12.6min acumulado
  Época  8/10 | Loss: 0.5379 | Val Acc: 80.01% | 14.4min acumulado
  Época  9/10 | Loss: 0.5399 | Val Acc: 80.97% | 16.2min acumulado
  Época 10/10 | Loss: 0.5358 | Val Acc: 80.11% | 18.1min acumulado
  ✅ 18.1 min total | Acurácia final: 80.11%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 4 — Fine-Tuning Parcial (layer4)
Descongela `layer4` com Discriminative LRs.

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
for p in m.layer4.parameters():
    p.requires_grad = True
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(
    [{'params': m.layer4.parameters(), 'lr': 1e-4},
     {'params': m.fc.parameters(),    'lr': 1e-3}],
    momentum=0.9)
run_experiment('Fine-Tuning Parcial (layer4)', m, opt, train_loader_std)


  Fine-Tuning Parcial (layer4)
  Parâmetros treináveis: 8,398,858
  Época  1/10 | Loss: 1.0860 | Val Acc: 78.18% | 1.9min acumulado
  Época  2/10 | Loss: 0.6246 | Val Acc: 81.19% | 3.7min acumulado
  Época  3/10 | Loss: 0.5329 | Val Acc: 82.56% | 5.6min acumulado
  Época  4/10 | Loss: 0.4816 | Val Acc: 83.48% | 7.4min acumulado
  Época  5/10 | Loss: 0.4470 | Val Acc: 84.12% | 9.3min acumulado
  Época  6/10 | Loss: 0.4217 | Val Acc: 84.59% | 11.1min acumulado
  Época  7/10 | Loss: 0.3986 | Val Acc: 85.17% | 13.0min acumulado
  Época  8/10 | Loss: 0.3803 | Val Acc: 85.77% | 14.9min acumulado
  Época  9/10 | Loss: 0.3640 | Val Acc: 86.07% | 16.7min acumulado
  Época 10/10 | Loss: 0.3476 | Val Acc: 86.52% | 18.6min acumulado
  ✅ 18.6 min total | Acurácia final: 86.52%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experimento 5 — Fine-Tuning Completo
Todas as 18 camadas abertas com `lr=0.001`.

In [ ]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.001, momentum=0.9)
run_experiment('Fine-Tuning Completo', m, opt, train_loader_std)


  Fine-Tuning Completo
  Parâmetros treináveis: 11,181,642
  Época  1/10 | Loss: 0.6700 | Val Acc: 90.59% | 2.7min acumulado
  Época  2/10 | Loss: 0.2331 | Val Acc: 92.60% | 5.4min acumulado
  Época  3/10 | Loss: 0.1564 | Val Acc: 93.64% | 8.1min acumulado
  Época  4/10 | Loss: 0.1123 | Val Acc: 93.88% | 10.8min acumulado
  Época  5/10 | Loss: 0.0800 | Val Acc: 94.15% | 13.4min acumulado
  Época  6/10 | Loss: 0.0575 | Val Acc: 94.09% | 16.1min acumulado
  Época  7/10 | Loss: 0.0411 | Val Acc: 94.47% | 18.8min acumulado
  Época  8/10 | Loss: 0.0289 | Val Acc: 94.26% | 21.5min acumulado
  Época  9/10 | Loss: 0.0218 | Val Acc: 94.41% | 24.2min acumulado
  Época 10/10 | Loss: 0.0163 | Val Acc: 94.58% | 26.9min acumulado
  ✅ 26.9 min total | Acurácia final: 94.58%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Tabela de Resultados

In [ ]:
print(f'\n{"="*65}')
print(f'{"BENCHMARK COMPLETO — CIFAR-10 · 50.000 imagens · 10 épocas":^65}')
print(f'{"="*65}')
print(f'{"Experimento":<30} {"Params":>12} {"Acurácia":>10} {"Tempo":>10}')
print(f'{"-"*65}')
for name, r in results.items():
    print(f'{name:<30} {r["params"]:>12,} {r["acc"]:>9.2f}% {r["time_min"]:>8.1f}min')
print(f'{"="*65}')
best = max(results, key=lambda k: results[k]['acc'])
print(f'  Melhor: {best} → {results[best]["acc"]:.2f}%')


   BENCHMARK COMPLETO — CIFAR-10 · 50.000 imagens · 10 épocas    
Experimento                          Params   Acurácia      Tempo
-----------------------------------------------------------------
Scratch Padrão                   11,181,642     75.97%     26.9min
Scratch + Augmentation           11,181,642     83.42%     27.1min
Feature Extraction                    5,130     80.11%     18.1min
Fine-Tuning Parcial (layer4)      8,398,858     86.52%     18.6min
Fine-Tuning Completo             11,181,642     94.58%     26.9min
  Melhor: Fine-Tuning Completo → 94.58%
